# 01 · SOFR curve calibration

Bootstrap a USD SOFR OIS discount/projection curve with QuantLib and test it:

1. Build the curve from synthetic OIS par quotes (`OISRateHelper`).
2. **Repricing test**: every input instrument must reprice to its quote.
3. **Independent check**: fresh `MakeOIS` swaps struck at the quoted par rate should have ~0 NPV.
4. **Interpolation comparison**: zero and 1M-forward curves under four bootstrap schemes.
5. **Node sensitivity**: how a 1bp bump in one input quote moves the zero curve.

Data is synthetic (`ql_helpers.SOFR_OIS_QUOTES`); replace with live quotes to calibrate to the market.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))   # run from notebooks/ so ql_helpers imports
import numpy as np
import QuantLib as ql
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import ql_helpers as h
import plot_utils

pio.renderers.default = "plotly_mimetype"
plot_utils.apply()
pd.options.display.float_format = "{:,.4f}".format
print("QuantLib", ql.__version__, "| valuation date", h.set_eval_date())

QuantLib 1.43 | valuation date September 25th, 2026


## 1. Bootstrap and repricing test

In [2]:
sofr = h.SofrCurve(interpolator="LogLinearDiscount")
rows = []
for tenor, helper in zip(sofr.quotes, sofr.helpers):
    rows.append({
        "tenor": tenor,
        "maturity": helper.maturityDate().ISO(),
        "quote_%": sofr.quotes[tenor].value() * 100,
        "implied_%": helper.impliedQuote() * 100,
        "error_bp": (helper.impliedQuote() - helper.quote().value()) * 1e4,
    })
repricing = pd.DataFrame(rows).set_index("tenor")
display(repricing)
assert repricing["error_bp"].abs().max() < 1e-6, "calibration does not reprice inputs"
print("max |error| (bp):", repricing["error_bp"].abs().max())

,maturity,quote_%,implied_%,error_bp
tenor,,,,
1W,2026-10-06,4.0500,4.0500,-0.0000
1M,2026-10-29,4.0300,4.0300,-0.0000
3M,2026-12-29,3.9800,3.9800,-0.0000
6M,2027-03-29,3.8800,3.8800,0.0000
1Y,2027-09-29,3.7200,3.7200,-0.0000
2Y,2028-09-29,3.6100,3.6100,-0.0000
3Y,2029-09-28,3.6200,3.6200,-0.0000
4Y,2030-09-30,3.6700,3.6700,0.0000
5Y,2031-09-29,3.7300,3.7300,0.0000


max |error| (bp): 2.1885271372923398e-10


## 2. Independent repricing with `MakeOIS`

Fresh swaps struck at the quoted par rate: fair rate should equal the quote and NPV should be ~0 (tiny residual from payment-lag conventions).

In [3]:
rows = []
for tenor, level in sofr.quote_levels.items():
    if tenor == "1W":   # sub-1M pillars are single-period OIS; skip in this check
        continue
    swap = ql.MakeOIS(ql.Period(tenor), sofr.index, level, ql.Period(0, ql.Days),
                      nominal=100e6, discountingTermStructure=sofr.handle, paymentLag=2)
    rows.append({"tenor": tenor, "NPV_usd": swap.NPV(), "fair_%": swap.fairRate() * 100,
                 "quote_%": level * 100, "diff_bp": (swap.fairRate() - level) * 1e4})
check = pd.DataFrame(rows).set_index("tenor")
display(check)
print("max |diff| (bp):", check["diff_bp"].abs().max())
assert check["diff_bp"].abs().max() < 1e-3

,NPV_usd,fair_%,quote_%,diff_bp
tenor,,,,
1M,-0.0000,4.0300,4.0300,-0.0000
3M,0.0000,3.9800,3.9800,0.0000
6M,0.0000,3.8800,3.8800,0.0000
1Y,0.0000,3.7200,3.7200,0.0000
2Y,-0.0000,3.6100,3.6100,-0.0000
3Y,-0.0000,3.6200,3.6200,-0.0000
4Y,0.0000,3.6700,3.6700,0.0000
5Y,0.0000,3.7300,3.7300,0.0000
7Y,0.0000,3.8500,3.8500,0.0000


max |diff| (bp): 2.20795604022328e-10


## 3. Interpolation schemes

Same inputs, different bootstrap interpolators. Zero rates agree closely, but forwards differ a lot: log-linear discount gives jagged forwards, cubic is smooth (and can overshoot).

In [4]:
curves = {name: h.SofrCurve(interpolator=name) for name in h.INTERPOLATORS}
ts = np.linspace(0.1, 30, 300)

def zero(c, t):  return c.curve.zeroRate(float(t), ql.Continuous).rate() * 100
def fwd1m(c, t): return c.curve.forwardRate(float(t), float(t) + 1/12, ql.Continuous).rate() * 100

fig = go.Figure()
for name, c in curves.items():
    fig.add_scatter(x=ts, y=[fwd1m(c, t) for t in ts], name=f"1M fwd · {name}")
fig.update_layout(title="1M forward rate by interpolation scheme", xaxis_title="years", yaxis_title="%", height=450)
fig.show()

tbl = pd.DataFrame({name: [zero(c, t) for t in (1, 2, 5, 10, 20, 30)] for name, c in curves.items()},
                   index=["1Y", "2Y", "5Y", "10Y", "20Y", "30Y"])
display(tbl)

,LogLinearDiscount,LinearZero,LogCubicDiscount,FlatForward
1Y,3.7088,3.7108,3.7096,3.7088
2Y,3.5962,3.5970,3.5959,3.5962
5Y,3.7185,3.7183,3.7183,3.7185
10Y,3.9989,3.9996,3.9994,3.9989
20Y,4.1886,4.1907,4.1880,4.1886
30Y,4.0759,4.0748,4.0705,4.0759


## 4. Node sensitivity

Bump one input quote by +1bp and look at the change in zero rates. Quotes are live `SimpleQuote`s, so the curve re-bootstraps automatically.

In [5]:
grid = [1, 2, 3, 5, 7, 10, 15, 20, 30]
base = [zero(sofr, t) for t in grid]
jac = {}
for pillar in ["1Y", "2Y", "5Y", "10Y", "30Y"]:
    sofr.bump(pillar, 1.0)
    jac[pillar] = [(zero(sofr, t) - b) * 100 for t, b in zip(grid, base)]   # bp
    sofr.reset()
jac = pd.DataFrame(jac, index=[f"{t}Y" for t in grid])
jac.columns.name = "bumped quote (+1bp)"
display(jac.style.format("{:.3f}").set_caption("Change in zero rate (bp)"))

fig = go.Figure([go.Scatter(x=grid, y=jac[c], name=f"bump {c}") for c in jac.columns])
fig.update_layout(title="Zero-rate response to +1bp in a single OIS quote", xaxis_title="years", yaxis_title="bp", height=400)
fig.show()

bumped quote (+1bp),1Y,2Y,5Y,10Y,30Y
1Y,0.956,0.000,0.000,0.000,0.000
2Y,-0.011,0.983,-0.000,-0.000,-0.000
3Y,-0.012,-0.017,0.000,0.000,0.000
5Y,-0.007,-0.015,1.042,0.000,0.000
7Y,-0.006,-0.011,-0.038,-0.000,-0.000
10Y,-0.004,-0.008,-0.033,1.132,0.000
15Y,-0.003,-0.006,-0.023,-0.132,-0.000
20Y,-0.002,-0.004,-0.018,-0.104,-0.000
30Y,-0.001,-0.003,-0.011,-0.067,1.548


## 5. Shocks and sanity checks

Shift all inputs and confirm the curve follows; check discount factors are monotone and forwards positive.

In [6]:
sofr.bump(None, 25.0)
print("10Y zero after +25bp on all quotes: %.4f%%  (was %.4f%%)" % (zero(sofr, 10), base[grid.index(10)]))
sofr.reset()
df = [sofr.curve.discount(float(t)) for t in ts]
assert all(a > b for a, b in zip(df, df[1:])), "discount factors not monotone"
fwds = [fwd1m(sofr, t) for t in ts]
print("forward range: %.3f%% .. %.3f%%" % (min(fwds), max(fwds)), "| negative forwards:", any(f < 0 for f in fwds))

10Y zero after +25bp on all quotes: 4.2444%  (was 3.9989%)
forward range: 3.483% .. 4.442% | negative forwards: False
